In [2]:
import pandas as pd

# Load raw data
df_raw = pd.read_csv("../data/raw/train_u6lujuX_CVtuZ9i.csv")

# 1. Basic info
print("=== INFO ===")
print(df_raw.info())

# 2. Shape
print("\n=== SHAPE ===")
print(df_raw.shape)

# 3. Data types
print("\n=== DATA TYPES ===")
print(df_raw.dtypes)

# 4. Missing values
print("\n=== MISSING VALUES ===")
print(df_raw.isnull().sum())

# 5. Unique values
print("\n=== UNIQUE VALUES ===")
for col in df_raw.columns:
    print(f"{col}: {df_raw[col].nunique()}")

# 6. Summary statistics (numerical)
print("\n=== NUMERICAL SUMMARY ===")
print(df_raw.describe())

# 7. Summary statistics (categorical)
print("\n=== CATEGORICAL SUMMARY ===")
print(df_raw.describe(include='object'))

=== INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB
None

=== SHAPE ===
(614, 13)

=== DATA TYPES ===
Loan_ID               object
Gender              

In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

# ── 1. Load Data ──────────────────────────────────
df = pd.read_csv("../data/processed/train_final.csv")

X_train = df.drop('Loan_Status', axis=1)
y_train = df['Loan_Status']

# ── 2. Train Random Forest ────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42
)

# ── 3. Stratified CV — same setup as Logistic Regression ──
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    rf_model, X_train, y_train,
    cv=cv,
    scoring=['f1', 'precision', 'recall', 'roc_auc']
)

# ── 4. Print Results ──────────────────────────────
print("── Random Forest CV Results ──")
print(f"  F1       : {cv_results['test_f1'].mean():.3f} ± {cv_results['test_f1'].std():.3f}")
print(f"  Precision: {cv_results['test_precision'].mean():.3f} ± {cv_results['test_precision'].std():.3f}")
print(f"  Recall   : {cv_results['test_recall'].mean():.3f} ± {cv_results['test_recall'].std():.3f}")
print(f"  ROC-AUC  : {cv_results['test_roc_auc'].mean():.3f} ± {cv_results['test_roc_auc'].std():.3f}")

print("\n── Comparison ──")
lr_f1 = 0.793  # your Logistic Regression baseline
rf_f1 = cv_results['test_f1'].mean()
diff = rf_f1 - lr_f1

print(f"  Logistic Regression F1 : {lr_f1:.3f}")
print(f"  Random Forest F1       : {rf_f1:.3f}")
print(f"  Difference             : {diff:+.3f}")

if diff > 0.02:
    print("\n✅ Random Forest wins — update model_trainer.py")
elif diff > 0:
    print("\n⚠️  Marginal improvement — stick with Logistic Regression")
else:
    print("\n❌ Logistic Regression wins — keep model_trainer.py as is")


# ── 5. Load Test Data ─────────────────────────────
test_df = pd.read_csv("../data/processed/test_final.csv")

X_test = test_df.drop('Loan_Status', axis=1)
y_test = test_df['Loan_Status']

# ── 6. Fit on full train, evaluate on test ────────
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

# ── 7. Test Set Metrics ───────────────────────────
from sklearn.metrics import (
    precision_score, recall_score, f1_score, 
    roc_auc_score, classification_report, confusion_matrix
)

print("── Random Forest Test Set Results ──")
print(f"  F1       : {f1_score(y_test, y_pred):.3f}")
print(f"  Precision: {precision_score(y_test, y_pred):.3f}")
print(f"  Recall   : {recall_score(y_test, y_pred):.3f}")
print(f"  ROC-AUC  : {roc_auc_score(y_test, y_prob):.3f}")

print("\n── Classification Report ──")
print(classification_report(y_test, y_pred, target_names=['Rejected', 'Approved']))

print("── Confusion Matrix ──")
cm = confusion_matrix(y_test, y_pred)
print(f"                 Predicted Rejected  Predicted Approved")
print(f"Actual Rejected       {cm[0][0]:<6}                {cm[0][1]:<6}")
print(f"Actual Approved       {cm[1][0]:<6}                {cm[1][1]:<6}")

── Random Forest CV Results ──
  F1       : 0.851 ± 0.028
  Precision: 0.780 ± 0.034
  Recall   : 0.938 ± 0.040
  ROC-AUC  : 0.751 ± 0.068

── Comparison ──
  Logistic Regression F1 : 0.793
  Random Forest F1       : 0.851
  Difference             : +0.058

✅ Random Forest wins — update model_trainer.py
── Random Forest Test Set Results ──
  F1       : 0.883
  Precision: 0.840
  Recall   : 0.929
  ROC-AUC  : 0.799

── Classification Report ──
              precision    recall  f1-score   support

    Rejected       0.79      0.61      0.69        38
    Approved       0.84      0.93      0.88        85

    accuracy                           0.83       123
   macro avg       0.82      0.77      0.78       123
weighted avg       0.83      0.83      0.82       123

── Confusion Matrix ──
                 Predicted Rejected  Predicted Approved
Actual Rejected       23                    15    
Actual Approved       6                     79    
